# Hybrid BiLSTM-CRF for Kannada NER
.

1. **Neural Embeddings**: Word and Character-level representations.
2. **Linguistic Features**: Explicit POS tagging and Gazetteer (lookup) features fed into the network.
3. **CRF Layer**: Conditional Random Field decoding to ensure valid label sequences.

| Feature | Purpose |
|---|---|
| Word Embedding | Global semantic meaning |
| Char CNN | Sub-word morphology (handles 'ನಲ್ಲಿ', 'ರಿಂದ', etc.) |
| POS Tags | Syntactic category (Noun, Verb, Proper Noun) |
| Gazetteers | Specialized lists (Districts, Honorifics) |
| CRF decoding | Sequence consistency (no I-PER after B-LOC) |

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install datasets==2.16.1 torch tqdm scikit-learn sympy pytorch-crf seqeval polyglot pyicu morfessor pycld2 --upgrade -q
!python -c "from polyglot.downloader import downloader; downloader.download('pos2.kn')"

print("Installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 27.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.

In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import json
import os
import numpy as np
from tqdm.auto import tqdm
from torchcrf import CRF
from seqeval.metrics import classification_report, f1_score
from polyglot.text import Text as PolyText
import copy

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using DEVICE: {DEVICE}")

Using DEVICE: cuda


In [ ]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────────
MAX_LEN = 100
MAX_WORD_LEN = 15
BATCH_SIZE = 32
WORD_EMB_DIM = 128
CHAR_EMB_DIM = 64
CHAR_CNN_FILTER = 64
HE_FEATURE_DIM = 32   # Dimension for hand-engineered feature projection
HIDDEN_DIM = 256
LR = 0.001
EPOCHS = 10
PATIENCE = 3

VOCAB_SAMPLES = 50000
TRAIN_SAMPLES = 200000 # Using 200k for faster training in this hybrid setup
VAL_SAMPLES = 5000

TAG_MAP = {"O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4, "B-LOC": 5, "I-LOC": 6}
INV_TAG_MAP = {v: k for k, v in TAG_MAP.items()}

In [ ]:
# ── Cell 4: Linguistic Gazetteers ─────────────────────────────────────────────
HONORIFICS = {'ಡಾ', 'ಶ್ರೀ', 'ಶ್ರೀಮತಿ', 'ಅವರು', 'ಅವರನ್ನು', 'ಗೌಡ', 'ಗೌಡರು'}
KARNATAKA_PLACES = {'ಬೆಂಗಳೂರು', 'ಮೈಸೂರು', 'ಹುಬ್ಬಳ್ಳಿ', 'ಧಾರವಾಡ', 'ಮಂಗಳೂರು', 'ಬೆಳಗಾವಿ'}
LOCATION_TYPE_WORDS = {'ನಗರ', 'ಹಳ್ಳಿ', 'ಗ್ರಾಮ', 'ಜಿಲ್ಲೆ', 'ತಾಲ್ಲೂಕು', 'ರಸ್ತೆ', 'ಮಾರ್ಗ'}
ORG_KEYWORDS = {'ಸರ್ಕಾರ', 'ಪ್ರಾಧಿಕಾರ', 'ಬ್ಯಾಂಕ್', 'ಆಸ್ಪತ್ರೆ', 'ಶಾಲೆ', 'ಕಾಲೇಜು'}
VIBHAKTI_SUFFIXES = ['ನಲ್ಲಿ', 'ರಲ್ಲಿ', 'ಕ್ಕೆ', 'ಗೆ', 'ರಿಂದ', 'ಇಂದ', 'ನ್ನು', 'ಅನ್ನು']

POS_MAP = {
    'NOUN': 0, 'PROPN': 1, 'VERB': 2, 'ADP': 3, 'ADJ': 4, 'ADV': 5,
    'PRON': 6, 'NUM': 7, 'DET': 8, 'CONJ': 9, 'PART': 10, 'X': 11
}

print("Gazetteers and POS map loaded.")

Gazetteers and POS map loaded.


In [ ]:
# ── Cell 5: Hybrid Vocabulary & Feature Builder ───────────────────────────────
def build_vocabs():
    print("Building Vocabulary (Vocab Samples: 50k)...")
    ds = load_dataset("ai4bharat/naamapadam", "kn", split="train", streaming=True, trust_remote_code=True)
    word_vocab = {"<PAD>": 0, "<UNK>": 1}
    char_vocab = {"<PAD>": 0, "<UNK>": 1}

    for row in tqdm(ds.take(VOCAB_SAMPLES), total=VOCAB_SAMPLES):
        for token in row['tokens']:
            if token not in word_vocab: word_vocab[token] = len(word_vocab)
            for char in token:
                if char not in char_vocab: char_vocab[char] = len(char_vocab)
    return word_vocab, char_vocab

W_VOCAB, C_VOCAB = build_vocabs()

Building Vocabulary (Vocab Samples: 50k)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  0%|          | 0/50000 [00:00<?, ?it/s]

In [ ]:
# ── Cell 6: POS Tagger (Morph-based for speed in DataLoader) ───────────────────
def get_morph_pos_idx(word):
    """Fast rule-based POS for DataLoader input."""
    if word.isdigit(): return POS_MAP['NUM']
    if any(word.endswith(v) for v in ['ತ್ತಾರೆ', 'ದ್ದಾರೆ', 'ತ್ತಿದೆ']): return POS_MAP['VERB']
    if any(word.endswith(v) for v in VIBHAKTI_SUFFIXES): return POS_MAP['NOUN']
    if word in HONORIFICS: return POS_MAP['PART']
    # Heuristic: Title-case or All-caps Latin is usually PROPN
    if any(c.isascii() and c.isupper() for c in word): return POS_MAP['PROPN']
    return POS_MAP['NOUN']

def get_engineered_features(word):
    """Convert linguistic features into a binary/categorical list."""
    feats = [
        float(word in HONORIFICS),
        float(word in KARNATAKA_PLACES),
        float(any(loc in word for loc in LOCATION_TYPE_WORDS)),
        float(any(org in word for org in ORG_KEYWORDS)),
        float(any(word.endswith(v) for v in VIBHAKTI_SUFFIXES)),
        float(any(c.isdigit() for c in word))
    ]
    return feats # Length 6

In [ ]:
# ── Cell 7: Dataset & DataLoader (Sequential Padding) ─────────────────────────
class HybridNERDataset(Dataset):
    def __init__(self, raw_dataset):
        self.data = list(raw_dataset)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        tokens = row['tokens'][:MAX_LEN]
        tags = row['ner_tags'][:MAX_LEN]

        word_ids = [W_VOCAB.get(t, W_VOCAB["<UNK>"]) for t in tokens]

        char_ids = []
        pos_ids = []
        he_features = []

        for t in tokens:
            # Chars
            c_ids = [C_VOCAB.get(c, C_VOCAB["<UNK>"]) for c in t[:MAX_WORD_LEN]]
            c_ids += [0] * (MAX_WORD_LEN - len(c_ids))
            char_ids.append(c_ids)

            # Linguistic Features
            pos_ids.append(get_morph_pos_idx(t))
            he_features.append(get_engineered_features(t))

        return (
            torch.tensor(word_ids),
            torch.tensor(char_ids),
            torch.tensor(pos_ids),
            torch.tensor(he_features, dtype=torch.float),
            torch.tensor(tags)
        )

def hybrid_collate(batch):
    words, chars, pos, he, tags = zip(*batch)

    mask = [torch.ones(len(w), dtype=torch.bool) for w in words]

    padded_words = pad_sequence(words, batch_first=True, padding_value=0)
    padded_pos   = pad_sequence(pos, batch_first=True, padding_value=0)
    padded_tags  = pad_sequence(tags, batch_first=True, padding_value=0)
    padded_mask  = pad_sequence(mask, batch_first=True, padding_value=False)

    # Pad characters and HE features manually for 3D tensors
    max_seq = padded_words.size(1)
    padded_chars = torch.zeros(len(chars), max_seq, MAX_WORD_LEN, dtype=torch.long)
    padded_he    = torch.zeros(len(he), max_seq, 6, dtype=torch.float)

    for i, seq in enumerate(chars):
        padded_chars[i, :seq.size(0), :] = seq
    for i, seq in enumerate(he):
        padded_he[i, :seq.size(0), :] = seq

    return padded_words, padded_chars, padded_pos, padded_he, padded_tags, padded_mask

print("DataLoader utility ready.")

DataLoader utility ready.


In [ ]:
# ── Cell 8: The SOTA Hybrid Model ─────────────────────────────────────────────
class HybridBiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, char_vocab_size, num_pos, he_dim, num_tags):
        super(HybridBiLSTM_CRF, self).__init__()

        # 1. Neural Embeddings
        self.word_embeddings = nn.Embedding(vocab_size, WORD_EMB_DIM, padding_idx=0)
        self.char_embeddings = nn.Embedding(char_vocab_size, CHAR_EMB_DIM, padding_idx=0)

        # 2. Phonological/subword Morphological pattern detector (Char-CNN) -
        self.char_cnn = nn.Conv1d(CHAR_EMB_DIM, CHAR_CNN_FILTER, kernel_size=3, padding=1)

        # 3. Linguistic Feature Projection (HE = Hand-Engineered explicitly)
        self.pos_embeddings = nn.Embedding(num_pos, 16)
        self.he_projection = nn.Linear(6 + 16, HE_FEATURE_DIM) # 6 binary + 16 pos_emb

        # 4. Sequence Encoder
        # Total input = Word(128) + Char(64) + Linguistic(32) = 224
        input_dim = WORD_EMB_DIM + CHAR_CNN_FILTER + HE_FEATURE_DIM
        self.lstm = nn.LSTM(input_dim, HIDDEN_DIM, bidirectional=True, batch_first=True)

        # 5. Output Decoder
        self.hidden2tag = nn.Linear(HIDDEN_DIM * 2, num_tags)
        self.crf = CRF(num_tags, batch_first=True)

    def _get_lstm_features(self, word_seq, char_seq, pos_seq, he_seq):
        batch, seq_len, word_len = char_seq.size()

        # a. Process Chars (Morphology)
        char_flat = char_seq.view(-1, word_len)
        char_embeds = self.char_embeddings(char_flat).transpose(1, 2) # (B*S, Dim, W_Len)
        char_cnn_out = self.char_cnn(char_embeds)
        char_cnn_out, _ = torch.max(char_cnn_out, dim=2) # Max-pool morphology
        char_features = char_cnn_out.view(batch, seq_len, -1)

        # b. Process Linguistic Features
        pos_embeds = self.pos_embeddings(pos_seq)
        he_combined = torch.cat([he_seq, pos_embeds], dim=2)
        ling_features = torch.relu(self.he_projection(he_combined))

        # c. Concatenate all streams
        word_embeds = self.word_embeddings(word_seq)
        combined = torch.cat([word_embeds, char_features, ling_features], dim=2)

        # d. Encode
        lstm_out, _ = self.lstm(combined)
        return self.hidden2tag(lstm_out)

    def neg_log_likelihood(self, word_seq, char_seq, pos_seq, he_seq, tags, mask):
        emissions = self._get_lstm_features(word_seq, char_seq, pos_seq, he_seq)
        return -self.crf(emissions, tags, mask=mask, reduction='mean')

    def forward(self, word_seq, char_seq, pos_seq, he_seq, mask):
        emissions = self._get_lstm_features(word_seq, char_seq, pos_seq, he_seq)
        return self.crf.decode(emissions, mask=mask)

print("Hybrid Model Architecture defined.")

Hybrid Model Architecture defined.


In [ ]:
# ── Cell 9: Load & Prepare Data ────────────────────────────────────────────────
print("Loading dataset splits...")
raw_ds = load_dataset("ai4bharat/naamapadam", "kn", split="train", streaming=True, trust_remote_code=True)

# Streaming to list for shuffling capability
ds_list = list(tqdm(raw_ds.take(TRAIN_SAMPLES + VAL_SAMPLES), total=TRAIN_SAMPLES + VAL_SAMPLES))
train_split = ds_list[:TRAIN_SAMPLES]
val_split   = ds_list[TRAIN_SAMPLES:]

train_loader = DataLoader(HybridNERDataset(train_split), batch_size=BATCH_SIZE, shuffle=True, collate_fn=hybrid_collate)
val_loader   = DataLoader(HybridNERDataset(val_split), batch_size=BATCH_SIZE, collate_fn=hybrid_collate)

print(f"Loaders created. Train: {len(train_split)} batches.")

Loading dataset splits...


  0%|          | 0/205000 [00:00<?, ?it/s]

Loaders created. Train: 200000 batches.


In [ ]:
# ── Cell 10: Training Loop with Early Stopping ───────────────────────────────
model = HybridBiLSTM_CRF(len(W_VOCAB), len(C_VOCAB), len(POS_MAP), 6, len(TAG_MAP)).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

def validate(model, loader):
    model.eval()
    true_tags = []
    pred_tags = []
    with torch.no_grad():
        for batch in loader:
            words, chars, pos, he, tags, mask = [b.to(DEVICE) for b in batch]
            preds = model(words, chars, pos, he, mask)

            # Unpack predictions and ground truth using mask
            for i in range(len(preds)):
                true_tags.append([INV_TAG_MAP[t.item()] for t in tags[i][mask[i]]])
                pred_tags.append([INV_TAG_MAP[p] for p in preds[i]])

    f1 = f1_score(true_tags, pred_tags)
    return f1

print(f"Starting training loop with Early Stopping (Max Epochs: {EPOCHS}, Patience: {PATIENCE})...\n")

best_f1 = -1
no_improve_count = 0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    total_loss = 0
    for i, batch in enumerate(pbar):
        words, chars, pos, he, tags, mask = [b.to(DEVICE) for b in batch]

        optimizer.zero_grad()
        loss = model.neg_log_likelihood(words, chars, pos, he, tags, mask)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if i % 10 == 0:
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    val_f1 = validate(model, val_loader)

    print(f"\nEpoch {epoch+1} Summary: Avg Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}")

    # Early Stopping Logic
    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve_count = 0
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"New best model found! (F1: {best_f1:.4f})")
    else:
        no_improve_count += 1
        print(f"No improvement for {no_improve_count} epoch(s).")

    if no_improve_count >= PATIENCE:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Best model state restored.")

Starting training loop with Early Stopping (Max Epochs: 10, Patience: 3)...



Epoch 1/10:   0%|          | 0/6250 [00:00<?, ?it/s]


Epoch 1 Summary: Avg Loss: 2.7304 | Val F1: 0.7127
New best model found! (F1: 0.7127)


Epoch 2/10:   0%|          | 0/6250 [00:00<?, ?it/s]


Epoch 2 Summary: Avg Loss: 1.7989 | Val F1: 0.7299
New best model found! (F1: 0.7299)


...


Epoch 3 Summary: Avg Loss: 1.5037 | Val F1: 0.7293
No improvement for 1 epoch(s).


Epoch 4/10:   0%|          | 0/6250 [00:00<?, ?it/s]


Epoch 4 Summary: Avg Loss: 1.2389 | Val F1: 0.7249
No improvement for 2 epoch(s).


Epoch 5/10:   0%|          | 0/6250 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from seqeval.metrics import accuracy_score

def evaluate_accuracy(model, loader):
    model.eval()
    true_tags, pred_tags = [], []

    with torch.no_grad():
        for batch in loader:
            # Move to device (GPU/CPU)
            words, chars, pos, he, tags, mask = [b.to(DEVICE) for b in batch]

            # Get Viterbi predictions from CRF
            preds = model(words, chars, pos, he, mask)

            # Map indices back to labels (excluding padding)
            for i in range(len(preds)):
                true_tags.append([INV_TAG_MAP[t.item()] for t in tags[i][mask[i]]])
                pred_tags.append([INV_TAG_MAP[p] for p in preds[i]])

    # Calculate Token-level Accuracy
    acc = accuracy_score(true_tags, pred_tags)
    print(f"Overall Token-level Accuracy: {acc:.4f}")
    return acc


evaluate_accuracy(model, val_loader)

Overall Token-level Accuracy: 0.9046


0.9046172663337071

In [ ]:
# ── Cell 12: Save Model ───────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'word_vocab': W_VOCAB,
    'char_vocab': C_VOCAB,
    'tag_map': TAG_MAP
}, 'hybrid_ner_bilstm_crf.pth')

print("✅ Saved Hybrid Model: hybrid_ner_bilstm_crf.pth")

✅ Saved Hybrid Model: hybrid_ner_bilstm_crf.pth


In [ ]:
# ── Cell 13: Hybrid Inference Test ────────────────────────────────────────────
def predict_hybrid(text):
    tokens = text.split()
    w_ids = torch.tensor([[W_VOCAB.get(t, 1) for t in tokens]]).to(DEVICE)

    c_ids = []
    p_ids = []
    h_ids = []
    for t in tokens:
        # Chars
        chars = [C_VOCAB.get(c, 1) for c in t[:MAX_WORD_LEN]]
        chars += [0] * (MAX_WORD_LEN - len(chars))
        c_ids.append(chars)
        # Linguistic
        p_ids.append(get_morph_pos_idx(t))
        h_ids.append(get_engineered_features(t))

    c_ids = torch.tensor([c_ids]).to(DEVICE)
    p_ids = torch.tensor([p_ids]).to(DEVICE)
    h_ids = torch.tensor([h_ids], dtype=torch.float).to(DEVICE)
    mask  = torch.ones((1, len(tokens)), dtype=torch.bool).to(DEVICE)

    model.eval()
    with torch.no_grad():
        preds = model(w_ids, c_ids, p_ids, h_ids, mask)[0]

    print(f"\nInput: {text}")
    for t, p in zip(tokens, preds):
        label = INV_TAG_MAP[p]
        flag = " ◀ ENTITY" if label != "O" else ""
        print(f"  {t:<20} : {label}{flag}")

predict_hybrid("ರಮೇಶ್ ಅವರು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಇರುವ ಬಿಬಿಎಂಪಿ ಕಚೇರಿಗೆ ಹೋಗಿ ಸಚಿವರನ್ನು ಭೇಟಿ ಮಾಡಿದರು")


Input: ರಮೇಶ್ ಅವರು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಇರುವ ಬಿಬಿಎಂಪಿ ಕಚೇರಿಗೆ ಹೋಗಿ ಸಚಿವರನ್ನು ಭೇಟಿ ಮಾಡಿದರು
  ರಮೇಶ್                : B-PER ◀ ENTITY
  ಅವರು                 : O
  ಬೆಂಗಳೂರಿನಲ್ಲಿ        : B-LOC ◀ ENTITY
  ಇರುವ                 : O
  ಬಿಬಿಎಂಪಿ             : O
  ಕಚೇರಿಗೆ              : O
  ಹೋಗಿ                 : O
  ಸಚಿವರನ್ನು            : O
  ಭೇಟಿ                 : O
  ಮಾಡಿದರು              : O


In [ ]:
predict_hybrid("ಮೈಸೂರಿನಲ್ಲಿ ಲಕ್ಷ್ಮಿ ಅವರ ಬ್ಯಾಂಕ್ ಖಾತೆಯಿಂದ ಹಣ ವರ್ಗಾವಣೆಯಾಗಿದೆ")

predict_hybrid("ಬೆಂಗಳೂರು ಮಹಾನಗರ ಪಾಲಿಕೆ ಅಧ್ಯಕ್ಷ ರಮೇಶ್ ಕುಮಾರ್ ಅವರು ಹೇಳಿದರು")

predict_hybrid("BBMP ಕಚೇರಿಯಲ್ಲಿ ಡಾ ಶ್ರೀಧರ್ ಅವರು 4ನೇ ಬ್ಲಾಕ್ ರಸ್ತೆ ದುರಸ್ತಿ ಬಗ್ಗೆ ಮಾತಾಡಿದರು")


Input: ಮೈಸೂರಿನಲ್ಲಿ ಲಕ್ಷ್ಮಿ ಅವರ ಬ್ಯಾಂಕ್ ಖಾತೆಯಿಂದ ಹಣ ವರ್ಗಾವಣೆಯಾಗಿದೆ
  ಮೈಸೂರಿನಲ್ಲಿ          : O
  ಲಕ್ಷ್ಮಿ              : B-PER ◀ ENTITY
  ಅವರ                  : O
  ಬ್ಯಾಂಕ್              : O
  ಖಾತೆಯಿಂದ             : O
  ಹಣ                   : O
  ವರ್ಗಾವಣೆಯಾಗಿದೆ       : O

Input: ಬೆಂಗಳೂರು ಮಹಾನಗರ ಪಾಲಿಕೆ ಅಧ್ಯಕ್ಷ ರಮೇಶ್ ಕುಮಾರ್ ಅವರು ಹೇಳಿದರು
  ಬೆಂಗಳೂರು             : B-ORG ◀ ENTITY
  ಮಹಾನಗರ               : I-ORG ◀ ENTITY
  ಪಾಲಿಕೆ               : I-ORG ◀ ENTITY
  ಅಧ್ಯಕ್ಷ              : O
  ರಮೇಶ್                : B-PER ◀ ENTITY
  ಕುಮಾರ್               : I-PER ◀ ENTITY
  ಅವರು                 : O
  ಹೇಳಿದರು              : O

Input: BBMP ಕಚೇರಿಯಲ್ಲಿ ಡಾ ಶ್ರೀಧರ್ ಅವರು 4ನೇ ಬ್ಲಾಕ್ ರಸ್ತೆ ದುರಸ್ತಿ ಬಗ್ಗೆ ಮಾತಾಡಿದರು
  BBMP                 : B-ORG ◀ ENTITY
  ಕಚೇರಿಯಲ್ಲಿ           : O
  ಡಾ                   : O
  ಶ್ರೀಧರ್              : B-PER ◀ ENTITY
  ಅವರು                 : O
  4ನೇ                  : O
  ಬ್ಲಾಕ್               : O
  ರಸ್ತೆ                : O
  ದುರಸ್ತಿ              : O
  ಬಗ್ಗೆ                : O
  ಮಾತಾಡಿ

In [ ]:
predict_hybrid("ಬೆಂಗಳೂರಿನ ಇಂದಿರಾ ಅವರ ತಾಯಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ಮಾರತಹಳ್ಳಿ ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.")


Input: ಬೆಂಗಳೂರಿನ ಇಂದಿರಾ ಅವರ ತಾಯಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ಮಾರತಹಳ್ಳಿ ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.
  ಬೆಂಗಳೂರಿನ            : B-LOC ◀ ENTITY
  ಇಂದಿರಾ               : O
  ಅವರ                  : O
  ತಾಯಿ                 : O
  ಸುಧಾ                 : B-PER ◀ ENTITY
  ಅವರ                  : O
  ಮನೆಗೆ                : O
  ನುಗ್ಗಿ               : O
  ಕಳ್ಳರು               : O
  ದರೋಡೆ                : O
  ಮಾಡಿದ್ದಾರೆ.          : O
  ಮಾರತಹಳ್ಳಿ            : B-LOC ◀ ENTITY
  ನಿವಾಸಿ               : O
  ಮಹೇಶ್                : B-PER ◀ ENTITY
  ಅವರು                 : O
  ಪೊಲೀಸ್               : O
  ಠಾಣೆಗೆ               : O
  ದೂರು                 : O
  ನೀಡಿದರು.             : O


In [ ]:
predict_hybrid("ಬೆಂಗಳೂರಿನ ಇಂದಿರಾನಗರದಲ್ಲಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.")


Input: ಬೆಂಗಳೂರಿನ ಇಂದಿರಾನಗರದಲ್ಲಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.
  ಬೆಂಗಳೂರಿನ            : B-LOC ◀ ENTITY
  ಇಂದಿರಾನಗರದಲ್ಲಿ       : B-LOC ◀ ENTITY
  ಸುಧಾ                 : O
  ಅವರ                  : O
  ಮನೆಗೆ                : O
  ನುಗ್ಗಿ               : O
  ಕಳ್ಳರು               : O
  ದರೋಡೆ                : O
  ಮಾಡಿದ್ದಾರೆ.          : O
  ನಿವಾಸಿ               : O
  ಮಹೇಶ್                : B-PER ◀ ENTITY
  ಅವರು                 : O
  ಪೊಲೀಸ್               : O
  ಠಾಣೆಗೆ               : O
  ದೂರು                 : O
  ನೀಡಿದರು.             : O
